# Phase 2: Inspect the Dataset

## Goals for this notebook:
- Load the final combined dataset
- Answer all inspection questions:
  1. How many rows?
  2. How many columns?
  3. Missing values?
  4. Duplicate posts?
  5. Empty posts?
  6. Average text length?
  7. Distribution by subreddit/platform?
  8. Date range?
  9. Which AI tools appear most often?

In [3]:
import json
import re
from collections import Counter
from pathlib import Path

import matplotlib
matplotlib.use('Agg')  # Non-interactive backend
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Set style for plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Create figures directory if it doesn't exist
figures_dir = Path('../figures')
figures_dir.mkdir(parents=True, exist_ok=True)

---
## Step 1: Load the Final Combined Dataset

In [4]:
# Load the dataset
combined_path = Path('../data/raw/combined_reddit_posts.jsonl')
posts = []
with open(combined_path, 'r', encoding='utf-8') as f:
    for line in f:
        posts.append(json.loads(line))
df = pd.DataFrame(posts)
print(f'✅ Dataset loaded successfully!')

✅ Dataset loaded successfully!


---
## Question 1: How many rows?

In [5]:
num_rows = len(df)
print(f'Number of rows (posts): {num_rows}')

Number of rows (posts): 5406


---
## Question 2: How many columns?

In [6]:
num_cols = len(df.columns)
print(f'Number of columns: {num_cols}')
print('\nColumn names:')
for col in df.columns:
    print(f'  - {col}')

Number of columns: 27

Column names:
  - sample_id
  - post_id
  - source_date
  - subreddit
  - created_at
  - score
  - model_mentions
  - selftext
  - sentiment_label
  - labeler_notes
  - source
  - id
  - title
  - num_comments
  - created_utc
  - created_iso
  - author
  - permalink
  - url
  - text
  - likes
  - replies_count
  - quotes_count
  - reblogs_count
  - prediction
  - confidence
  - relevant


---
## Question 3: Missing values?

In [5]:
missing = df.isnull().sum().sort_values(ascending=False)
print('Missing values per column:')
print(missing[missing > 0])
if missing.sum() == 0:
    print('\n✅ No missing values in the dataset!')
else:
    print(f'\n❌ Total missing values: {missing.sum()}')

Missing values per column:
prediction         5265
relevant           5265
reblogs_count      5265
confidence         5265
likes              5265
quotes_count       5265
replies_count      5265
text               5238
sample_id          3406
post_id            3406
source_date        3406
sentiment_label    3406
labeler_notes      3406
model_mentions     3406
created_at         3406
url                2156
permalink          2141
created_iso        2141
author             2141
title              2141
num_comments       2141
id                 2141
created_utc        2141
subreddit           141
score               141
selftext              4
dtype: int64

❌ Total missing values: 83364


---
## Question 4: Duplicate posts?

In [6]:
# Build a stable key across sources (HuggingFace rows often lack `id`)
df['dedupe_key'] = (
    df['id'].astype('string')
    .fillna(df['post_id'].astype('string'))
    .fillna(df['sample_id'].astype('string'))
)

# Important: missing keys must NOT count as duplicates of each other
dup_mask = df['dedupe_key'].notna() & df.duplicated(subset=['dedupe_key'], keep='first')
duplicates = int(dup_mask.sum())
missing_keys = int(df['dedupe_key'].isna().sum())

print(f'Number of duplicate posts (by id/post_id/sample_id): {duplicates}')
print(f'Rows with no usable ID key (left untouched): {missing_keys}')

if duplicates == 0:
    print('\n✅ No duplicate posts!')
else:
    df = df.loc[~dup_mask].copy()
    print(f'\nRemoved {duplicates} duplicates, remaining rows: {len(df)}')

Number of duplicate posts (by id/post_id/sample_id): 0
Rows with no usable ID key (left untouched): 141

✅ No duplicate posts!


---
## Question 5: Empty posts?
First, let's create a combined text field (title + selftext) for all posts:

In [7]:
# Create combined text field (sources use title/selftext and/or text)
df['title'] = df['title'].fillna('').astype(str)
df['selftext'] = df['selftext'].fillna('').astype(str)
df['text'] = df['text'].fillna('').astype(str)
df['full_text'] = (
    df['title'].str.strip() + ' ' + df['selftext'].str.strip() + ' ' + df['text'].str.strip()
).str.strip()
df['full_text'] = df['full_text'].str.replace(r'\s+', ' ', regex=True)

# Check empty posts
empty_posts = df[df['full_text'] == '']
print(f'Number of empty posts (no title or body): {len(empty_posts)}')
if len(empty_posts) > 0:
    print(f'\nRemoving {len(empty_posts)} empty posts...')
    df = df[df['full_text'] != ''].copy()
    print(f'Remaining rows: {len(df)}')
else:
    print('\n✅ No empty posts!')

Number of empty posts (no title or body): 0

✅ No empty posts!


---
## Question 6: Average text length?

In [8]:
df['text_length'] = df['full_text'].str.len()
avg_length = df['text_length'].mean()
median_length = df['text_length'].median()
min_length = df['text_length'].min()
max_length = df['text_length'].max()

print(f'Average text length: {avg_length:.2f} characters')
print(f'Median text length: {median_length:.2f} characters')
print(f'Shortest post: {min_length} characters')
print(f'Longest post: {max_length} characters')

# Plot distribution
plt.figure(figsize=(12, 6))
plt.hist(df['text_length'], bins=50, edgecolor='black')
plt.title('Distribution of Post Text Lengths')
plt.xlabel('Text Length (characters)')
plt.ylabel('Number of Posts')
plt.yscale('log')  # Use log scale for better visibility
plt.tight_layout()
plt.savefig(figures_dir / 'text_length_distribution.png', dpi=300, bbox_inches='tight')
print(f'✅ Plot saved to: {figures_dir / "text_length_distribution.png"}')

Average text length: 1822.34 characters
Median text length: 1133.00 characters
Shortest post: 8 characters
Longest post: 56405 characters


✅ Plot saved to: ../figures/text_length_distribution.png


---
## Question 7: Distribution by subreddit/platform?

In [7]:
# Standardize platform/subreddit names for better grouping
def get_platform(row):
    subreddit = row.get('subreddit')
    source = str(row.get('source') or '')

    if pd.notna(subreddit):
        subreddit = str(subreddit)
        lower = subreddit.lower()
        if 'github' in lower:
            return 'GitHub Issues'
        if 'stackoverflow' in lower:
            return 'Stack Overflow'
        if 'hackernews' in lower:
            return 'Hacker News'
        return f'Reddit: r/{subreddit}'

    # Fallback when subreddit is missing
    source_l = source.lower()
    if 'github' in source_l:
        return 'GitHub Issues'
    if 'stackoverflow' in source_l:
        return 'Stack Overflow'
    if 'hackernews' in source_l or source_l == 'hackernews':
        return 'Hacker News'
    if source:
        return f'Source: {source}'
    return 'Unknown'

df['platform'] = df.apply(get_platform, axis=1)
platform_counts = df['platform'].value_counts().sort_values(ascending=True)

print('Distribution by platform/subreddit:')
print(platform_counts)

# Plot
plt.figure(figsize=(12, 8))
platform_counts.plot(kind='barh', edgecolor='black')
plt.title('Number of Posts by Platform/Subreddit')
plt.xlabel('Number of Posts')
plt.ylabel('Platform/Subreddit')
plt.tight_layout()
plt.savefig(figures_dir / 'posts_by_platform.png', dpi=300, bbox_inches='tight')
print(f'✅ Plot saved to: {figures_dir / "posts_by_platform.png"}')

Distribution by platform/subreddit:
platform
Reddit: r/ExperiencedDevs                      17
Reddit: r/programming                          59
Reddit: r/webdev                               65
Reddit: r/MachineLearning                      69
Reddit: r/cscareerquestions                    95
Source: huggingface_divde-sentiment_posts     141
Reddit: r/artificial                          178
Hacker News                                   236
Reddit: r/singularity                         397
Reddit: r/OpenAI                              444
Reddit: r/LocalLLaMA                          981
Stack Overflow                               1108
GitHub Issues                                1616
Name: count, dtype: int64
✅ Plot saved to: ../figures/posts_by_platform.png


---
## Question 8: Date range?

In [10]:
# Coalesce timestamps across sources (created_iso / created_at / created_utc / source_date)
created = pd.to_datetime(df['created_iso'], errors='coerce', utc=True)
created = created.fillna(pd.to_datetime(df['created_at'], errors='coerce', utc=True))
created = created.fillna(pd.to_datetime(df['source_date'], errors='coerce', utc=True))
created = created.fillna(pd.to_datetime(df['created_utc'], unit='s', errors='coerce', utc=True))
df['created_datetime'] = created

min_date = df['created_datetime'].min()
max_date = df['created_datetime'].max()
missing_dates = df['created_datetime'].isna().sum()

print(f'Earliest post: {min_date}')
print(f'Latest post: {max_date}')
if pd.notna(min_date) and pd.notna(max_date):
    print(f'\nDate range: {max_date - min_date}')
print(f'Posts with no parseable date: {missing_dates}')

# Convert to timezone-naive UTC before to_period (avoids UserWarning)
ts = df['created_datetime']
if getattr(ts.dt, 'tz', None) is not None:
    ts = ts.dt.tz_convert('UTC').dt.tz_localize(None)
df['created_month'] = ts.dt.to_period('M')
monthly_counts = df['created_month'].value_counts().sort_index()

plt.figure(figsize=(12, 6))
monthly_counts.plot(kind='line', marker='o')
plt.title('Number of Posts Over Time')
plt.xlabel('Month')
plt.ylabel('Number of Posts')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(figures_dir / 'posts_over_time.png', dpi=300, bbox_inches='tight')
print(f'✅ Plot saved to: {figures_dir / "posts_over_time.png"}')

Earliest post: 2005-02-04 22:16:00+00:00
Latest post: 2026-07-22 00:25:20+00:00

Date range: 7837 days 02:09:20
Posts with no parseable date: 141


✅ Plot saved to: ../figures/posts_over_time.png


---
## Question 9: Which AI tools appear most often?

In [11]:
# Define AI tools to look for (longer names first so regex prefers them)
ai_tools = [
    'GitHub Copilot', 'VS Code Copilot', 'Stable Diffusion', 'ChatGPT',
    'CodeLlama', 'Starcoder', 'AutoGPT', 'BabyAGI', 'Midjourney', 'DALL-E',
    'LLaMA 2', 'LLaMA', 'Llama', 'Claude', 'Copilot', 'Cursor', 'Gemini',
    'Mistral', 'Agents', 'Agent', 'GPT',
]

# Create a regex pattern (case-insensitive); sort by length so "GitHub Copilot" wins over "Copilot"
pattern = '|'.join(re.escape(tool) for tool in sorted(ai_tools, key=len, reverse=True))

# Find all mentions
mentions = []
for text in df['full_text']:
    matches = re.findall(pattern, text, flags=re.IGNORECASE)
    # Normalize to consistent capitalization
    for match in matches:
        for tool in ai_tools:
            if match.lower() == tool.lower():
                mentions.append(tool)
                break

# Count mentions
tool_counts = Counter(mentions).most_common(15)
print('Top 15 most mentioned AI tools:')
for tool, count in tool_counts:
    print(f'  {tool}: {count} mentions')

# Plot
if tool_counts:
    tools, counts = zip(*tool_counts)
    plt.figure(figsize=(12, 6))
    plt.bar(list(tools), list(counts), edgecolor='black')
    plt.title('Top 15 Most Mentioned AI Tools')
    plt.xlabel('AI Tool')
    plt.ylabel('Number of Mentions')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(figures_dir / 'ai_tool_mentions.png', dpi=300, bbox_inches='tight')
    print(f'✅ Plot saved to: {figures_dir / "ai_tool_mentions.png"}')
else:
    print('\n⚠️ No AI tool mentions found to plot.')

Top 15 most mentioned AI tools:
  LLaMA: 1557 mentions
  Agent: 1174 mentions
  Claude: 999 mentions
  Gemini: 859 mentions
  GPT: 679 mentions
  Copilot: 629 mentions
  Agents: 488 mentions
  ChatGPT: 419 mentions
  Cursor: 187 mentions
  Mistral: 101 mentions
  GitHub Copilot: 85 mentions
  CodeLlama: 31 mentions
  VS Code Copilot: 6 mentions
  Stable Diffusion: 5 mentions
  LLaMA 2: 5 mentions


✅ Plot saved to: ../figures/ai_tool_mentions.png


---
## Step 2: Save the cleaned dataset (optional but recommended)
Since we've done some cleaning (removed duplicates, empty posts, added useful fields), let's save a cleaned version:

In [12]:
cleaned_path = Path('../data/processed/combined_posts_cleaned.jsonl')
cleaned_path.parent.mkdir(parents=True, exist_ok=True)

# Drop helper columns (Period/tz datetime objects are awkward in JSON)
df_to_save = df.drop(
    columns=['created_datetime', 'created_month', 'dedupe_key'],
    errors='ignore',
)

# date_format='iso' avoids Pandas4Warning about deprecated epoch defaults
df_to_save.to_json(
    cleaned_path,
    orient='records',
    lines=True,
    force_ascii=False,
    date_format='iso',
)

print(f'✅ Cleaned dataset saved to: {cleaned_path}')
print(f'Rows written: {len(df_to_save)}')

✅ Cleaned dataset saved to: ../data/processed/combined_posts_cleaned.jsonl
Rows written: 5406
